# Notebook 2: Feature Engineering
## Federal Reserve Interest Rate Prediction

**Goal:** Transform 8 raw economic indicators into 67 informative features by creating:
1. **Lag features** — capture delayed economic effects (1, 3, 6, 12 months)
2. **Rolling statistics** — smooth short-term volatility (3 & 6-month windows)
3. **Interaction features** — Phillips Curve proxy, growth rates
4. **Classification target** — rate change direction (Increase/Decrease/No Change)


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2196F3','#F44336','#4CAF50','#FF9800','#9C27B0',
           '#00BCD4','#E91E63','#795548','#607D8B','#FF5722']
sns.set_palette(PALETTE)

DATA_PATH = r"d:/Projects/ML website/ML-Project/App/Tabs/Datasets/finaldataset.csv"
OUT_PATH  = r"d:/Projects/ML website/ML-Project/ml_analysis/outputs"


In [ ]:
# Load and clean data
df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
# Basic cleaning
df['ConsumerPriceIndexAllItems'] = df['ConsumerPriceIndexAllItems'].replace(0, np.nan)
df = df.ffill().bfill()
for col in df.select_dtypes(include=np.number).columns:
    p01, p99 = df[col].quantile(0.01), df[col].quantile(0.99)
    df[col] = df[col].clip(p01, p99)
df = df.set_index('date')
print(f"Starting shape: {df.shape}")

FEATURES = ['ConsumerPriceIndexAllItems','GDP','InflationConsumerPrice',
            'MedianConsumerPriceIndex','RealGDP','RealGDPPerCapita',
            'RealPotentialGDP','UnemployemenrRate']


## 1. Classification Target — Rate Direction

In [ ]:
# Create rate change direction labels
df['RateChange']   = df['FEDRates'].diff()
df['RateDirection'] = 'No_Change'
df.loc[df['RateChange'] >  0.05, 'RateDirection'] = 'Increase'
df.loc[df['RateChange'] < -0.05, 'RateDirection'] = 'Decrease'

print("Class distribution:")
print(df['RateDirection'].value_counts())
print(f"\nClass proportions:")
print(df['RateDirection'].value_counts(normalize=True).round(3))


In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
vc = df['RateDirection'].value_counts()
colors = ['#F44336','#4CAF50','#FF9800'][:len(vc)]
axes[0].bar(vc.index, vc.values, color=colors)
for i, (k, v) in enumerate(vc.items()):
    axes[0].text(i, v+1, str(v), ha='center', fontweight='bold')
axes[0].set_title('Rate Direction Count', fontweight='bold')
axes[1].pie(vc.values, labels=vc.index, autopct='%1.1f%%', colors=colors)
axes[1].set_title('Rate Direction Proportion', fontweight='bold')
plt.tight_layout(); plt.show()


## 2. Lag Features

In [ ]:
# Create lag features (1, 3, 6, 12 months)
lag_months = [1, 3, 6, 12]
for col in FEATURES:
    for lag in lag_months:
        df[f'{col}_lag{lag}'] = df[col].shift(lag)

print(f"Shape after lag features: {df.shape}")
print(f"New lag features created: {len(FEATURES) * len(lag_months)}")


In [ ]:
# Correlation of lag features with FED Rate
from scipy.stats import pearsonr
lag_corrs = {}
for col in FEATURES:
    for lag in lag_months:
        feat_name = f'{col}_lag{lag}'
        r, p = pearsonr(df[[feat_name, 'FEDRates']].dropna()[feat_name],
                        df[[feat_name, 'FEDRates']].dropna()['FEDRates'])
        lag_corrs[feat_name] = {'r': round(r,4), 'p': round(p,6)}

lag_corr_df = pd.DataFrame(lag_corrs).T.sort_values('r', key=abs, ascending=False)
print("Top 10 lag features by |correlation|:")
display(lag_corr_df.head(10))


## 3. Rolling Statistics

In [ ]:
# Rolling mean and standard deviation (3 and 6 month windows)
for col in FEATURES:
    df[f'{col}_roll3_mean'] = df[col].rolling(3).mean()
    df[f'{col}_roll6_mean'] = df[col].rolling(6).mean()
    df[f'{col}_roll3_std']  = df[col].rolling(3).std()

print(f"Shape after rolling features: {df.shape}")

# Visualize rolling vs raw for InflationConsumerPrice
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(df.index, df['InflationConsumerPrice'], alpha=0.4, color='#2196F3', label='Raw')
ax.plot(df.index, df['InflationConsumerPrice_roll3_mean'], linewidth=2,
        color='#F44336', label='3-month rolling mean')
ax.plot(df.index, df['InflationConsumerPrice_roll6_mean'], linewidth=2,
        color='#4CAF50', label='6-month rolling mean')
ax.set_title('Inflation: Raw vs Rolling Mean', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()


## 4. Interaction & Growth Features

In [ ]:
# Interaction features
df['Inflation_x_Unemployment'] = (df['InflationConsumerPrice'] * df['UnemployemenrRate'])
df['GDP_growth']     = df['GDP'].pct_change() * 100
df['RealGDP_growth'] = df['RealGDP'].pct_change() * 100

# Calendar features
df['Month']   = df.index.month
df['Year']    = df.index.year
df['Quarter'] = df.index.quarter

# Drop NaN rows from lag/rolling
n_before = len(df)
df = df.dropna()
print(f"Rows: {n_before} → {len(df)} (after dropping lag-NaN)")
print(f"Final feature count: {df.shape[1]}")


In [ ]:
# Visualize Phillips Curve proxy
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc = axes[0].scatter(df['UnemployemenrRate'], df['InflationConsumerPrice'],
                     c=df['FEDRates'], cmap='RdYlGn', s=20, alpha=0.6)
axes[0].set_xlabel('Unemployment Rate'); axes[0].set_ylabel('Inflation')
axes[0].set_title('Phillips Curve (color = FED Rate)', fontweight='bold')
plt.colorbar(sc, ax=axes[0], label='FED Rate %')

axes[1].scatter(df['GDP_growth'], df['FEDRates'], alpha=0.4, color='#2196F3', s=15)
axes[1].set_xlabel('GDP Growth Rate (%)'); axes[1].set_ylabel('FED Rate (%)')
axes[1].set_title('GDP Growth vs FED Rate', fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# Top 20 features by correlation with FED Rate
model_df = df.select_dtypes(include=np.number).dropna()
corr_with_target = model_df.corr()['FEDRates'].drop('FEDRates').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
top20 = corr_with_target.head(20)
colors_bar = ['#F44336' if v < 0 else '#2196F3' for v in top20.values]
ax.barh(range(len(top20)), top20.values, color=colors_bar)
ax.set_yticks(range(len(top20))); ax.set_yticklabels(top20.index, fontsize=9)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 20 Features — Correlation with FED Rate', fontweight='bold')
ax.set_xlabel('Pearson r')
plt.tight_layout(); plt.show()


## Summary
- 67 features created from 8 raw indicators
- Lag features (especially 12-month inflation lags) have the strongest correlation
- Lasso regression later confirms ~55% of features can be zeroed out